In [10]:
import importlib
import sys
from pathlib import Path

# Add the repository's src directory to the import path, regardless of the notebook's cwd.
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

import similarity_querying
importlib.reload(similarity_querying)

from similarity_querying import *

print("Imported similarity_querying from:", similarity_querying.__file__)

Imported similarity_querying from: /Users/anshbhatnagar/Desktop/DS3/ds3-example-project/src/similarity_querying.py


In [11]:
# Cell 2: Create real Trefle client

from pathlib import Path

token = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    env_path = candidate / ".env"
    if env_path.exists():
        for line in env_path.read_text().splitlines():
            if line.startswith("TREFLE_API_KEY="):
                token = line.split("=", 1)[1].strip()
                break
        if token:
            break

os.environ["TREFLE_TOKEN"] = token

client = TrefleClient()

print("TrefleClient initialized.")

TrefleClient initialized.


In [12]:
# Cell 3: Lightweight notebook test runner

import traceback
from typing import Dict, Any, List

TEST_RESULTS = []

def run_test(name, fn):
    try:
        result = fn()
        TEST_RESULTS.append({
            "test_name": name,
            "status": "PASS",
            "message": "",
        })
        print(f"PASS: {name}")
        return result
    except AssertionError as e:
        TEST_RESULTS.append({
            "test_name": name,
            "status": "FAIL",
            "message": str(e),
        })
        print(f"FAIL: {name}")
        print(" ", e)
    except Exception as e:
        TEST_RESULTS.append({
            "test_name": name,
            "status": "ERROR",
            "message": f"{type(e).__name__}: {e}",
        })
        print(f"ERROR: {name}")
        print(" ", type(e).__name__, e)
        traceback.print_exc(limit=1)


def show_test_summary():
    import pandas as pd

    df = pd.DataFrame(TEST_RESULTS)

    print("\n=== TEST SUMMARY ===")
    print(df["status"].value_counts())

    display(df)

In [13]:
# Cell 4: Shared assertion helpers

def assert_similarity_response_shape(result: Dict[str, Any]):
    assert isinstance(result, dict), "Result should be a dict"
    assert "basis" in result, "Missing basis"
    assert "count" in result, "Missing count"
    assert "results" in result, "Missing results"
    assert "warnings" in result, "Missing warnings"
    assert isinstance(result["results"], list), "results should be a list"
    assert isinstance(result["warnings"], list), "warnings should be a list"
    assert result["count"] == len(result["results"]), "count should equal len(results)"


def assert_max_results_respected(result: Dict[str, Any], max_results: int):
    assert result["count"] <= max_results, f"Returned more than max_results={max_results}"


def assert_image_only_respected(result: Dict[str, Any]):
    for plant in result["results"]:
        assert plant.get("image_url"), f"Plant missing image_url: {plant}"


def print_group_preview(result: Dict[str, Any], limit: int = 5):
    print("basis:", result.get("basis"))
    print("count:", result.get("count"))
    print("warnings:", result.get("warnings"))

    for plant in result.get("results", [])[:limit]:
        print(
            "-",
            plant.get("common_name"),
            "|",
            plant.get("scientific_name"),
            "| image:",
            bool(plant.get("image_url")),
        )

In [14]:
# Cell 5: Utility/helper function tests

def test_slugify():
    assert slugify("Vaccinium Corymbosum") == "vaccinium-corymbosum"
    assert slugify("Family_Name") == "family-name"
    assert slugify(None) is None


def test_unique_preserve_order():
    assert unique_preserve_order(["a", "b", "a", "c", "b"]) == ["a", "b", "c"]


def test_json_loads_safe():
    valid = json_loads_safe('{"hello": "world"}')
    invalid = json_loads_safe("{bad json")

    assert valid["hello"] == "world"
    assert invalid["error"] is True


def test_plant_identity():
    assert plant_identity({"id": 123, "slug": "abc"}) == "123"
    assert plant_identity({"slug": "abc"}) == "abc"
    assert plant_identity({"scientific_name": "Test plant"}) == "Test plant"


def test_has_image():
    assert has_image({"image_url": "https://example.com/x.jpg"}) is True
    assert has_image({"image_url": None, "images": {"leaf": [{"x": 1}]}}) is True
    assert has_image({"image_url": None, "images": {"leaf": []}}) is False


run_test("slugify", test_slugify)
run_test("unique_preserve_order", test_unique_preserve_order)
run_test("json_loads_safe", test_json_loads_safe)
run_test("plant_identity", test_plant_identity)
run_test("has_image", test_has_image)

PASS: slugify
PASS: unique_preserve_order
PASS: json_loads_safe
PASS: plant_identity
PASS: has_image


In [15]:
# Cell 6: Real API search/detail tests

def test_search_plant_blueberry():
    plant = search_plant(client, "blueberry")

    assert plant is not None, "Expected blueberry search to return a plant"
    assert plant.get("id") or plant.get("slug"), "Search result should have id or slug"
    assert plant.get("scientific_name"), "Search result should have scientific_name"

    print(plant)


def test_get_plant_details_blueberry():
    plant = get_plant_details(client, "blueberry")

    assert plant is not None, "Expected plant details for blueberry"
    assert plant.get("id") or plant.get("slug"), "Plant details should have id or slug"
    assert plant.get("main_species") is not None, "Plant details should include main_species"

    print("common_name:", plant.get("common_name"))
    print("scientific_name:", plant.get("scientific_name"))
    print("genus:", plant.get("genus"))
    print("family:", plant.get("family"))


def test_get_plant_details_slug():
    plant = get_plant_details(client, "senecio-gamolepis")

    assert plant is not None, "Expected plant details for senecio-gamolepis"
    assert plant.get("slug") == "senecio-gamolepis"

    print("scientific_name:", plant.get("scientific_name"))


def test_get_plant_details_bad_query():
    plant = get_plant_details(client, "asdfghjkl-not-a-real-plant-hopefully")

    assert plant is None, "Expected nonsense query to return None"


run_test("search_plant blueberry", test_search_plant_blueberry)
run_test("get_plant_details blueberry", test_get_plant_details_blueberry)
run_test("get_plant_details known slug", test_get_plant_details_slug)
run_test("get_plant_details bad query", test_get_plant_details_bad_query)

{'id': 111017, 'common_name': 'European blueberry', 'slug': 'vaccinium-myrtillus', 'scientific_name': 'Vaccinium myrtillus', 'year': 1753, 'bibliography': 'Sp. Pl.: 349 (1753)', 'author': 'L.', 'status': 'accepted', 'rank': 'species', 'family_common_name': None, 'genus_id': 5599, 'image_url': 'https://bs.plantnet.org/image/o/76e8f509a667a1110611d76c7b84d5bbc03ccb1f', 'synonyms': ['Vitis-idaea myrtillus', 'Vaccinium myrtillus var. minoriflora'], 'genus': 'Vaccinium', 'family': 'Ericaceae', 'links': {'self': '/api/v1/species/vaccinium-myrtillus', 'plant': '/api/v1/plants/vaccinium-myrtillus', 'genus': '/api/v1/genus/vaccinium'}}
PASS: search_plant blueberry
common_name: European blueberry
scientific_name: Vaccinium myrtillus
genus: {'id': 5599, 'name': 'Vaccinium', 'slug': 'vaccinium', 'links': {'self': '/api/v1/genus/vaccinium', 'plants': '/api/v1/genus/vaccinium/plants', 'species': '/api/v1/genus/vaccinium/species', 'family': '/api/v1/families/ericaceae'}}
family: {'id': 127, 'name': '

In [16]:
# Cell 7: Real API extraction tests

def test_extract_fields_blueberry():
    plant = get_plant_details(client, "blueberry")
    assert plant is not None

    genus_slug = extract_genus_slug(plant)
    family_slug = extract_family_slug(plant)
    family_name = extract_family_name(plant)
    distribution_slugs = extract_distribution_slugs(plant)

    assert genus_slug, "Expected genus slug"
    assert family_slug, "Expected family slug"
    assert family_name, "Expected family name"
    assert isinstance(distribution_slugs, list), "distribution_slugs should be list"

    print("genus_slug:", genus_slug)
    print("family_slug:", family_slug)
    print("family_name:", family_name)
    print("distribution_slugs:", distribution_slugs[:10])


def test_normalize_source_plant_blueberry():
    plant = get_plant_details(client, "blueberry")
    source = normalize_source_plant(plant)

    assert source["id"] or source["slug"], "source plant should have id or slug"
    assert source["scientific_name"], "source plant should have scientific_name"
    assert source["genus"], "source plant should have genus"
    assert source["family"], "source plant should have family"

    print(source)


run_test("extract fields blueberry", test_extract_fields_blueberry)
run_test("normalize_source_plant blueberry", test_normalize_source_plant_blueberry)

genus_slug: vaccinium
family_slug: ericaceae
family_name: Ericaceae
distribution_slugs: ['alb', 'abt', 'alt', 'amu', 'ari', 'aut', 'blt', 'blr', 'bgm', 'brc']
PASS: extract fields blueberry
{'id': 111062, 'slug': 'vaccinium-myrtillus', 'common_name': 'European blueberry', 'scientific_name': 'Vaccinium myrtillus', 'image_url': 'https://bs.plantnet.org/image/o/76e8f509a667a1110611d76c7b84d5bbc03ccb1f', 'genus': 'vaccinium', 'family': 'ericaceae'}
PASS: normalize_source_plant blueberry


In [17]:
# Cell 8: Real API genus similarity tests

def test_similar_by_genus_blueberry():
    max_results = 5

    result = get_similar_by_genus(
        client,
        "blueberry",
        max_results=max_results,
        image_only=False,
    )

    assert_similarity_response_shape(result)
    assert_max_results_respected(result, max_results)
    assert result["basis"] == "genus"
    assert result.get("genus_slug"), "Expected genus_slug"

    print_group_preview(result)


def test_similar_by_genus_plinia():
    max_results = 5

    result = get_similar_by_genus(
        client,
        "plinia",
        max_results=max_results,
        image_only=False,
    )

    assert_similarity_response_shape(result)
    assert_max_results_respected(result, max_results)
    assert result["basis"] == "genus"

    print_group_preview(result)


def test_similar_by_genus_image_only():
    max_results = 5

    result = get_similar_by_genus(
        client,
        "blueberry",
        max_results=max_results,
        image_only=True,
    )

    assert_similarity_response_shape(result)
    assert_max_results_respected(result, max_results)
    assert_image_only_respected(result)

    print_group_preview(result)


run_test("similar by genus blueberry", test_similar_by_genus_blueberry)
run_test("similar by genus plinia", test_similar_by_genus_plinia)
run_test("similar by genus image_only", test_similar_by_genus_image_only)

basis: genus
count: 5
warnings: []
- European blueberry | Vaccinium myrtillus | image: True
- Cowberry | Vaccinium vitis-idaea | image: True
- Bog blueberry | Vaccinium uliginosum | image: True
- European cranberry | Vaccinium oxycoccos | image: True
- American blueberry | Vaccinium corymbosum | image: True
PASS: similar by genus blueberry
basis: genus
count: 5
warnings: []
- None | Plinia rivularis | image: True
- Brazilian grapetree | Plinia cauliflora | image: True
- None | Plinia cordifolia | image: False
- None | Plinia edulis | image: True
- None | Plinia peruviana | image: False
PASS: similar by genus plinia
basis: genus
count: 5
warnings: []
- European blueberry | Vaccinium myrtillus | image: True
- Cowberry | Vaccinium vitis-idaea | image: True
- Bog blueberry | Vaccinium uliginosum | image: True
- European cranberry | Vaccinium oxycoccos | image: True
- American blueberry | Vaccinium corymbosum | image: True
PASS: similar by genus image_only


In [18]:
# Cell 9: Real API family similarity tests

def test_similar_by_family_blueberry():
    max_results = 5

    result = get_similar_by_family(
        client,
        "blueberry",
        max_results=max_results,
        image_only=False,
        exclude_genus_results=True,
    )

    assert_similarity_response_shape(result)
    assert_max_results_respected(result, max_results)
    assert result["basis"] == "family"
    assert result.get("family_slug"), "Expected family_slug"
    assert result.get("family_name"), "Expected family_name"

    source_genus = result.get("excluded_genus")
    if source_genus:
        for plant in result["results"]:
            if plant.get("genus"):
                assert plant.get("genus").lower() != source_genus.lower(), (
                    f"Family result should exclude source genus {source_genus}: {plant}"
                )

    print_group_preview(result)


def test_similar_by_family_image_only():
    max_results = 5

    result = get_similar_by_family(
        client,
        "blueberry",
        max_results=max_results,
        image_only=True,
        exclude_genus_results=True,
    )

    assert_similarity_response_shape(result)
    assert_max_results_respected(result, max_results)
    assert_image_only_respected(result)

    print_group_preview(result)


run_test("similar by family blueberry", test_similar_by_family_blueberry)
run_test("similar by family image_only", test_similar_by_family_image_only)

basis: family
count: 5
warnings: []
- Heather | Calluna vulgaris | image: True
- Curlew-berry | Empetrum nigrum | image: True
- Bog-heather | Erica tetralix | image: True
- Tree heath | Erica arborea | image: True
- Bog-rosemary | Andromeda polifolia | image: True
PASS: similar by family blueberry
basis: family
count: 5
warnings: []
- Heather | Calluna vulgaris | image: True
- Curlew-berry | Empetrum nigrum | image: True
- Bog-heather | Erica tetralix | image: True
- Tree heath | Erica arborea | image: True
- Bog-rosemary | Andromeda polifolia | image: True
PASS: similar by family image_only


In [19]:
# Cell 10: Real API distribution similarity tests

def test_similar_by_distribution_senecio():
    max_results = 5

    result = get_similar_by_distribution(
        client,
        "senecio-gamolepis",
        max_results=max_results,
        image_only=False,
        exclude_genus_and_family_results=True,
        max_zones_to_search=2,
    )

    assert_similarity_response_shape(result)
    assert_max_results_respected(result, max_results)
    assert result["basis"] == "distribution"

    if result["count"] > 0:
        assert "matched_distribution_slug" in result["results"][0]

    print_group_preview(result)


def test_similar_by_distribution_blueberry():
    max_results = 5

    result = get_similar_by_distribution(
        client,
        "blueberry",
        max_results=max_results,
        image_only=False,
        exclude_genus_and_family_results=True,
        max_zones_to_search=2,
    )

    assert_similarity_response_shape(result)
    assert_max_results_respected(result, max_results)
    assert result["basis"] == "distribution"

    print_group_preview(result)


def test_similar_by_distribution_image_only():
    max_results = 5

    result = get_similar_by_distribution(
        client,
        "senecio-gamolepis",
        max_results=max_results,
        image_only=True,
        exclude_genus_and_family_results=True,
        max_zones_to_search=2,
    )

    assert_similarity_response_shape(result)
    assert_max_results_respected(result, max_results)
    assert_image_only_respected(result)

    print_group_preview(result)


run_test("similar by distribution senecio", test_similar_by_distribution_senecio)
run_test("similar by distribution blueberry", test_similar_by_distribution_blueberry)
run_test("similar by distribution image_only", test_similar_by_distribution_image_only)

basis: distribution
count: 5
warnings: []
- Barnyard grass | Dactylis glomerata | image: True
- Narrow-leaf plantain | Plantago lanceolata | image: True
- Dutch clover | Trifolium repens | image: True
- Yorkshire-fog | Holcus lanatus | image: True
- Creeping buttercup | Ranunculus repens | image: True
PASS: similar by distribution senecio
basis: distribution
count: 5
warnings: []
- Common nettle | Urtica dioica | image: True
- Barnyard grass | Dactylis glomerata | image: True
- Narrow-leaf plantain | Plantago lanceolata | image: True
- Milfoil | Achillea millefolium | image: True
- Dutch clover | Trifolium repens | image: True
PASS: similar by distribution blueberry
basis: distribution
count: 5
warnings: []
- Barnyard grass | Dactylis glomerata | image: True
- Narrow-leaf plantain | Plantago lanceolata | image: True
- Dutch clover | Trifolium repens | image: True
- Yorkshire-fog | Holcus lanatus | image: True
- Creeping buttercup | Ranunculus repens | image: True
PASS: similar by distr

In [ ]:
# Cell 11: Real API trait similarity tests
# These are intentionally tolerant because Trefle often has null trait fields.

def test_similar_by_edible_part_blueberry():
    max_results = 5

    result = get_similar_by_edible_part(
        client,
        "blueberry",
        max_results=max_results,
        image_only=False,
    )

    assert_similarity_response_shape(result)
    assert_max_results_respected(result, max_results)
    assert result["basis"] == "edible_part"

    print_group_preview(result)


def test_similar_by_growth_habit_maple():
    max_results = 5

    result = get_similar_by_growth_habit(
        client,
        "maple",
        max_results=max_results,
        image_only=False,
    )

    assert_similarity_response_shape(result)
    assert_max_results_respected(result, max_results)
    assert result["basis"] == "growth_habit"

    print_group_preview(result)


def test_similar_by_growth_form_maple():
    max_results = 5

    result = get_similar_by_growth_form(
        client,
        "maple",
        max_results=max_results,
        image_only=False,
    )

    assert_similarity_response_shape(result)
    assert_max_results_respected(result, max_results)
    assert result["basis"] == "growth_form"

    print_group_preview(result)


def test_similar_by_fruit_color_blueberry():
    max_results = 5

    result = get_similar_by_fruit_color(
        client,
        "blueberry",
        max_results=max_results,
        image_only=False,
    )

    assert_similarity_response_shape(result)
    assert_max_results_respected(result, max_results)
    assert result["basis"] == "fruit_color"

    print_group_preview(result)


run_test("similar by edible_part blueberry", test_similar_by_edible_part_blueberry)
run_test("similar by growth_habit maple", test_similar_by_growth_habit_maple)
run_test("similar by growth_form maple", test_similar_by_growth_form_maple)
run_test("similar by fruit_color blueberry", test_similar_by_fruit_color_blueberry)

basis: edible_part
count: 0
warnings: ['No source value found for edible_part.']
PASS: similar by edible_part tomato
basis: growth_habit
count: 5
warnings: []
- Pedunculate oak | Quercus robur | image: True
- European ash | Fraxinus excelsior | image: True
- Beech | Fagus sylvatica | image: True
- Scotch pine | Pinus sylvestris | image: True
- Cork oak | Quercus suber | image: True
PASS: similar by growth_habit maple
basis: growth_form
count: 5
warnings: []
- Hawthorn | Crataegus monogyna | image: True
- Quickbeam | Sorbus aucuparia | image: True
- Scotch pine | Pinus sylvestris | image: True
- Sycamore maple | Acer pseudoplatanus | image: True
- Common spruce | Picea abies | image: True
PASS: similar by growth_form maple
basis: fruit_color
count: 0
warnings: ['No source value found for fruit_color.']
PASS: similar by fruit_color tomato


In [21]:
# Cell 12: Real API bundle tests

def test_bundle_blueberry_structure():
    max_results = 3

    bundle = get_similar_plants_bundle(
        client,
        "blueberry",
        max_results_per_group=max_results,
        image_only=False,
    )

    assert isinstance(bundle, dict)
    assert bundle["query"] == "blueberry"
    assert bundle["image_only"] is False
    assert "groups" in bundle
    assert isinstance(bundle["groups"], dict)

    expected_groups = {
        "same_genus",
        "same_family_excluding_genus",
        "same_distribution_excluding_taxonomy",
        "same_edible_part",
        "same_growth_habit",
        "same_growth_form",
        "same_fruit_color",
    }

    missing = expected_groups - set(bundle["groups"].keys())
    assert not missing, f"Missing bundle groups: {missing}"

    for group_name, group in bundle["groups"].items():
        print("\n---", group_name, "---")
        assert_similarity_response_shape(group)
        assert_max_results_respected(group, max_results)
        print_group_preview(group, limit=3)


def test_bundle_image_only():
    max_results = 3

    bundle = get_similar_plants_bundle(
        client,
        "blueberry",
        max_results_per_group=max_results,
        image_only=True,
    )

    for group_name, group in bundle["groups"].items():
        assert_similarity_response_shape(group)
        assert_max_results_respected(group, max_results)
        assert_image_only_respected(group)


def test_bundle_bad_query():
    bundle = get_similar_plants_bundle(
        client,
        "asdfghjkl-not-a-real-plant-hopefully",
        max_results_per_group=3,
        image_only=False,
    )

    assert "groups" in bundle

    for group_name, group in bundle["groups"].items():
        assert_similarity_response_shape(group)
        assert group["count"] == 0
        assert group["results"] == []
        assert group["warnings"], f"{group_name} should include warnings"


run_test("bundle blueberry structure", test_bundle_blueberry_structure)
run_test("bundle image_only", test_bundle_image_only)
run_test("bundle bad query", test_bundle_bad_query)


--- same_genus ---
basis: genus
count: 3
warnings: []
- European blueberry | Vaccinium myrtillus | image: True
- Cowberry | Vaccinium vitis-idaea | image: True
- Bog blueberry | Vaccinium uliginosum | image: True

--- same_family_excluding_genus ---
basis: family
count: 3
warnings: []
- Heather | Calluna vulgaris | image: True
- Curlew-berry | Empetrum nigrum | image: True
- Bog-heather | Erica tetralix | image: True

--- same_distribution_excluding_taxonomy ---
basis: distribution
count: 3
warnings: []
- Common nettle | Urtica dioica | image: True
- Barnyard grass | Dactylis glomerata | image: True
- Narrow-leaf plantain | Plantago lanceolata | image: True

--- same_edible_part ---
basis: edible_part
count: 0
warnings: ['No plant found for query: blueberry']

--- same_growth_habit ---
basis: growth_habit
count: 0
warnings: ['No plant found for query: blueberry']

--- same_growth_form ---
basis: growth_form
count: 0
warnings: ['No plant found for query: blueberry']

--- same_fruit_col

In [22]:
# Cell 13: Display one clean bundle result for manual inspection

sample_bundle = get_similar_plants_bundle(
    client,
    "blueberry",
    max_results_per_group=5,
    image_only=True,
)

for group_name, group in sample_bundle["groups"].items():
    print("\n==============================")
    print(group_name)
    print("==============================")
    print_group_preview(group, limit=5)


same_genus
basis: genus
count: 5
warnings: []
- European blueberry | Vaccinium myrtillus | image: True
- Cowberry | Vaccinium vitis-idaea | image: True
- Bog blueberry | Vaccinium uliginosum | image: True
- European cranberry | Vaccinium oxycoccos | image: True
- American blueberry | Vaccinium corymbosum | image: True

same_family_excluding_genus
basis: family
count: 5
warnings: []
- Heather | Calluna vulgaris | image: True
- Curlew-berry | Empetrum nigrum | image: True
- Bog-heather | Erica tetralix | image: True
- Tree heath | Erica arborea | image: True
- Bog-rosemary | Andromeda polifolia | image: True

same_distribution_excluding_taxonomy
basis: distribution
count: 5
warnings: []
- Common nettle | Urtica dioica | image: True
- Barnyard grass | Dactylis glomerata | image: True
- Narrow-leaf plantain | Plantago lanceolata | image: True
- Milfoil | Achillea millefolium | image: True
- Dutch clover | Trifolium repens | image: True

same_edible_part
basis: edible_part
count: 0
warning

In [23]:
# Cell 14: Final test summary

show_test_summary()


=== TEST SUMMARY ===
status
PASS    26
Name: count, dtype: int64


,test_name,status,message
0,slugify,PASS,
1,unique_preserve_order,PASS,
2,json_loads_safe,PASS,
3,plant_identity,PASS,
4,has_image,PASS,
5,search_plant blueberry,PASS,
6,get_plant_details blueberry,PASS,
7,get_plant_details known slug,PASS,
8,get_plant_details bad query,PASS,
9,extract fields blueberry,PASS,


In [33]:
# Cell: Image extraction + display helpers

from IPython.display import display, HTML
import html


def get_best_image_url(plant):
    """
    Extract best available image URL from a full Trefle plant detail object
    or a normalized/search plant card.
    """
    if not plant:
        return None

    if plant.get("image_url"):
        return plant["image_url"]

    main_species = plant.get("main_species") or {}
    if main_species.get("image_url"):
        return main_species["image_url"]

    images = main_species.get("images") or {}
    if isinstance(images, dict):
        for image_type, image_list in images.items():
            if not image_list:
                continue

            first = image_list[0]
            if isinstance(first, dict):
                return (
                    first.get("image_url")
                    or first.get("url")
                    or first.get("original_url")
                )

            if isinstance(first, str):
                return first

    return None


def image_url_is_valid(url):
    """
    Frontend-style sanity check.
    Do not use strict HEAD checks because some image hosts block them.
    """
    return isinstance(url, str) and url.startswith(("http://", "https://"))


def plant_display_name(plant):
    return (
        plant.get("common_name")
        or plant.get("scientific_name")
        or plant.get("slug")
        or "Unknown plant"
    )


def display_plant_image_card(plant, width=260):
    """
    Render a frontend-style plant image card in the notebook.
    Uses placeholder UI if no image exists.
    """
    image_url = get_best_image_url(plant)
    title = html.escape(str(plant_display_name(plant)))
    scientific_name = html.escape(str(plant.get("scientific_name") or ""))
    slug = html.escape(str(plant.get("slug") or ""))

    if image_url:
        image_html = f"""
            <img
                src="{html.escape(image_url)}"
                width="{width - 24}"
                style="border-radius: 8px; object-fit: cover; max-height: 220px;"
            />
        """
    else:
        image_html = f"""
            <div style="
                width: {width - 24}px;
                height: 180px;
                border-radius: 8px;
                background: #f2f2f2;
                display: flex;
                align-items: center;
                justify-content: center;
                color: #777;
                font-size: 14px;
            ">
                No image available
            </div>
        """

    display(HTML(f"""
        <div style="
            width: {width}px;
            border: 1px solid #ddd;
            border-radius: 12px;
            padding: 12px;
            margin: 12px 0;
            font-family: system-ui, -apple-system, BlinkMacSystemFont, sans-serif;
        ">
            {image_html}
            <div style="font-weight: 700; margin-top: 10px;">{title}</div>
            <div style="font-style: italic; color: #555; font-size: 14px;">{scientific_name}</div>
            <div style="color: #777; font-size: 12px; margin-top: 4px;">{slug}</div>
        </div>
    """))

In [36]:
# Cell: Test 1 — specific selected plant/species image extraction

def test_selected_species_image_displayable():
    """
    This tests the profile-page image case:
    user selected a specific species/plant slug, and we display its best image.
    """
    candidate_slugs = [
        "vaccinium-corymbosum",
        "solanum-lycopersicum",
        "cocos-nucifera",
        "helianthus-annuus",
        "rosa-rubiginosa",
        "fragaria-vesca",
        "monstera-deliciosa",
    ]

    selected = None
    selected_slug = None
    selected_image = None

    for slug in candidate_slugs:
        plant = get_plant_details(client, slug)
        image_url = get_best_image_url(plant)

        if plant and image_url_is_valid(image_url):
            selected = plant
            selected_slug = slug
            selected_image = image_url

        assert selected is not None, "No selected species with image found from candidate slugs"
        assert selected_image, "Expected selected species to have image"
        assert image_url_is_valid(selected_image), f"Invalid image URL: {selected_image}"

        print("Selected slug:", selected_slug)
        print("common_name:", selected.get("common_name"))
        print("scientific_name:", selected.get("scientific_name"))
        print("image_url:", selected_image)

        display_plant_image_card(selected)


run_test("selected species image displayable", test_selected_species_image_displayable)

Selected slug: vaccinium-corymbosum
common_name: American blueberry
scientific_name: Vaccinium corymbosum
image_url: https://bs.plantnet.org/image/o/0add97cba731c09cebc5a29b40d7819243195ca1


Selected slug: solanum-lycopersicum
common_name: Tomato
scientific_name: Solanum lycopersicum
image_url: https://bs.plantnet.org/image/o/400851a79391dbe6f667c66e4bf70299e9921853


Selected slug: cocos-nucifera
common_name: Coconut
scientific_name: Cocos nucifera
image_url: https://bs.plantnet.org/image/o/a0c4045351cd7e31ed6c4a338c4fd7f0be5c1a2f


Selected slug: helianthus-annuus
common_name: Sunflower
scientific_name: Helianthus annuus
image_url: https://bs.plantnet.org/image/o/e92bdbc5adc4e91cc0c5aa8e9f102833a28bc6e3


Selected slug: rosa-rubiginosa
common_name: Eglantine
scientific_name: Rosa rubiginosa
image_url: https://bs.plantnet.org/image/o/90aead7560b1ef01c807e287f30172add5fcae27


Selected slug: fragaria-vesca
common_name: Alpine strawberry
scientific_name: Fragaria vesca
image_url: https://bs.plantnet.org/image/o/4f45fd2d82661996f5d5a5613b39bdd1287a56bc


Selected slug: monstera-deliciosa
common_name: Fruit-salad-plant
scientific_name: Monstera deliciosa
image_url: https://bs.plantnet.org/image/o/a3a043c163463e4cf1941e473a134700ed327d84


PASS: selected species image displayable


In [41]:
# Cell: Test 2 — search results can produce image cards

def test_search_results_include_displayable_image_card():
    """
    This tests the search-results page case:
    user searches a common query, and at least one returned card can show an image.
    """
    seed_queries = [
        "tomato",
        "coconut",
        "sunflower",
        "rose",
        "strawberry",
        "banana",
        "maple",
        "oak",
        "monstera",
    ]

    selected_query = None
    selected_plant = None

    for query in seed_queries:
        payload = client.get("/plants/search", {"q": query})
        results = payload.get("data") or []

        for plant in results:
            image_url = get_best_image_url(plant)
            if image_url_is_valid(image_url):
                selected_query = query
                selected_plant = normalize_plant_card(plant)


        assert selected_plant is not None, "No search result with image found"
        assert selected_plant.get("id") or selected_plant.get("slug"), "Search card needs id or slug"
        assert selected_plant.get("scientific_name"), "Search card needs scientific_name"
        assert image_url_is_valid(selected_plant.get("image_url")), "Search card needs valid image_url"

        print("Selected query:", selected_query)
        print("Search result card:")
        print(selected_plant)

        display_plant_image_card(selected_plant)


run_test("search results include displayable image card", test_search_results_include_displayable_image_card)

Selected query: tomato
Search result card:
{'id': 269798, 'slug': 'solanum-triflorum', 'common_name': 'Cut-leaf nightshade', 'scientific_name': 'Solanum triflorum', 'family': 'Solanaceae', 'family_common_name': None, 'genus': 'Solanum', 'genus_id': 12343, 'image_url': 'https://bs.plantnet.org/image/o/4ca22331d8e62057364aa3397dc91a8e5d4c625b', 'rank': 'species', 'status': 'accepted', 'vegetable': None, 'edible': None, 'links': {'self': '/api/v1/species/solanum-triflorum', 'plant': '/api/v1/plants/solanum-triflorum', 'genus': '/api/v1/genus/solanum'}}


Selected query: coconut
Search result card:
{'id': 53992, 'slug': 'jubaeopsis-caffra', 'common_name': None, 'scientific_name': 'Jubaeopsis caffra', 'family': 'Arecaceae', 'family_common_name': None, 'genus': 'Jubaeopsis', 'genus_id': 2645, 'image_url': 'https://storage.googleapis.com/powop-assets/palmweb/palm_tc_105174_3_fullsize.jpg', 'rank': 'species', 'status': 'accepted', 'vegetable': None, 'edible': None, 'links': {'self': '/api/v1/species/jubaeopsis-caffra', 'plant': '/api/v1/plants/jubaeopsis-caffra', 'genus': '/api/v1/genus/jubaeopsis'}}


Selected query: sunflower
Search result card:
{'id': 23667, 'slug': 'helianthus-heterophyllus', 'common_name': 'Wetland sunflower', 'scientific_name': 'Helianthus heterophyllus', 'family': 'Asteraceae', 'family_common_name': 'Aster family', 'genus': 'Helianthus', 'genus_id': 130, 'image_url': 'https://bs.plantnet.org/image/o/716addb37cf8883bffe882d8e729f559291269a1', 'rank': 'species', 'status': 'accepted', 'vegetable': None, 'edible': None, 'links': {'self': '/api/v1/species/helianthus-heterophyllus', 'plant': '/api/v1/plants/helianthus-heterophyllus', 'genus': '/api/v1/genus/helianthus'}}


Selected query: rose
Search result card:
{'id': 150498, 'slug': 'bauera-rubioides', 'common_name': 'River-rose', 'scientific_name': 'Bauera rubioides', 'family': 'Cunoniaceae', 'family_common_name': None, 'genus': 'Bauera', 'genus_id': 7764, 'image_url': 'https://bs.plantnet.org/image/o/9b9f86e98d0d9a767debac957c9bb73b990c8021', 'rank': 'species', 'status': 'accepted', 'vegetable': None, 'edible': None, 'links': {'self': '/api/v1/species/bauera-rubioides', 'plant': '/api/v1/plants/bauera-rubioides', 'genus': '/api/v1/genus/bauera'}}


Selected query: strawberry
Search result card:
{'id': 183704, 'slug': 'euonymus-americanus', 'common_name': 'Bursting-heart', 'scientific_name': 'Euonymus americanus', 'family': 'Celastraceae', 'family_common_name': None, 'genus': 'Euonymus', 'genus_id': 7009, 'image_url': 'https://bs.plantnet.org/image/o/87b66b3f45707ac0d39a8e88a87f82401476b22c', 'rank': 'species', 'status': 'accepted', 'vegetable': None, 'edible': None, 'links': {'self': '/api/v1/species/euonymus-americanus', 'plant': '/api/v1/plants/euonymus-americanus', 'genus': '/api/v1/genus/euonymus'}}


Selected query: banana
Search result card:
{'id': 35967, 'slug': 'inga-affinis', 'common_name': None, 'scientific_name': 'Inga affinis', 'family': 'Fabaceae', 'family_common_name': None, 'genus': 'Inga', 'genus_id': 1933, 'image_url': 'https://bs.plantnet.org/image/o/b678801fa07519c470f6e47c762413ad3e2be049', 'rank': 'species', 'status': 'accepted', 'vegetable': None, 'edible': None, 'links': {'self': '/api/v1/species/inga-affinis', 'plant': '/api/v1/plants/inga-affinis', 'genus': '/api/v1/genus/inga'}}


Selected query: maple
Search result card:
{'id': 137960, 'slug': 'acer-rufinerve', 'common_name': 'Grey-bud snakebark maple', 'scientific_name': 'Acer rufinerve', 'family': 'Sapindaceae', 'family_common_name': None, 'genus': 'Acer', 'genus_id': 7255, 'image_url': 'https://bs.plantnet.org/image/o/1481c1b2f38140e01795e03476322aec0d56d84c', 'rank': 'species', 'status': 'accepted', 'vegetable': None, 'edible': None, 'links': {'self': '/api/v1/species/acer-rufinerve', 'plant': '/api/v1/plants/acer-rufinerve', 'genus': '/api/v1/genus/acer'}}


Selected query: oak
Search result card:
{'id': 77180, 'slug': 'quercus-stellata', 'common_name': 'Post oak', 'scientific_name': 'Quercus stellata', 'family': 'Fagaceae', 'family_common_name': None, 'genus': 'Quercus', 'genus_id': 3519, 'image_url': 'https://bs.plantnet.org/image/o/8fd1b424eb5b0108420dc108c636100728278ebc', 'rank': 'species', 'status': 'accepted', 'vegetable': None, 'edible': None, 'links': {'self': '/api/v1/species/quercus-stellata', 'plant': '/api/v1/plants/quercus-stellata', 'genus': '/api/v1/genus/quercus'}}


Selected query: monstera
Search result card:
{'id': 62433, 'slug': 'monstera-glaucescens', 'common_name': None, 'scientific_name': 'Monstera glaucescens', 'family': 'Araceae', 'family_common_name': None, 'genus': 'Monstera', 'genus_id': 2961, 'image_url': 'https://bs.plantnet.org/image/o/73eb1e3529266d15568bcbfa5ad8f920dba4bc09', 'rank': 'species', 'status': 'accepted', 'vegetable': None, 'edible': None, 'links': {'self': '/api/v1/species/monstera-glaucescens', 'plant': '/api/v1/plants/monstera-glaucescens', 'genus': '/api/v1/genus/monstera'}}


PASS: search results include displayable image card


In [42]:
# Cell: Test 3 — image_only=True contract for similarity results

def test_similarity_image_only_contract():
    """
    This tests the related-plants image contract:
    if image_only=True, every returned related plant should have image_url.
    It does NOT require every query to return images.
    """
    queries = [
        "blueberry",
        "tomato",
        "coconut",
        "maple",
        "monstera",
        "plinia",
    ]

    for query in queries:
        result = get_similar_by_genus(
            client,
            query,
            max_results=8,
            image_only=True,
        )

        assert_similarity_response_shape(result)

        for plant in result["results"]:
            assert plant.get("image_url"), (
                f"image_only=True returned imageless plant for query={query}: {plant}"
            )
            assert image_url_is_valid(plant["image_url"]), (
                f"Invalid image_url for query={query}: {plant['image_url']}"
            )

        print(f"{query}: {result['count']} image-bearing genus results")


run_test("similarity image_only contract", test_similarity_image_only_contract)

blueberry: 8 image-bearing genus results
tomato: 8 image-bearing genus results
coconut: 1 image-bearing genus results
maple: 8 image-bearing genus results
monstera: 8 image-bearing genus results
plinia: 8 image-bearing genus results
PASS: similarity image_only contract


In [44]:
# Cell: Test 4 — find and render one image-bearing similarity result

def test_similarity_result_image_card_displayable_when_available():
    """
    This tries multiple queries until it finds one related-plant result with an image,
    then renders it as a card.
    """
    seed_queries = [
        "tomato",
        "coconut",
        "rose",
        "sunflower",
        "strawberry",
        "maple",
        "oak",
        "banana",
        "monstera",
        "blueberry",
    ]

    selected_query = None
    selected_group = None
    selected_plant = None

    for query in seed_queries:
        result = get_similar_by_genus(
            client,
            query,
            max_results=10,
            image_only=True,
        )

        if result["results"]:
            selected_query = query
            selected_group = result
            selected_plant = result["results"][0]

        if selected_plant is None:
            print("No image-bearing genus similarity result found from seed queries.")
            print("This is not necessarily a failure; Trefle image coverage is sparse.")
            return

        assert_similarity_response_shape(selected_group)
        assert selected_plant.get("image_url")
        assert image_url_is_valid(selected_plant["image_url"])

        print("Selected query:", selected_query)
        print("basis:", selected_group["basis"])
        print("common_name:", selected_plant.get("common_name"))
        print("scientific_name:", selected_plant.get("scientific_name"))
        print("image_url:", selected_plant.get("image_url"))

        display_plant_image_card(selected_plant)


run_test("similarity result image card displayable when available", test_similarity_result_image_card_displayable_when_available)

Selected query: tomato
basis: genus
common_name: Bitter nightshade
scientific_name: Solanum dulcamara
image_url: https://bs.plantnet.org/image/o/104fdfd9f869285afe86b9ce59b53059953e4a5e


Selected query: coconut
basis: genus
common_name: Coconut
scientific_name: Cocos nucifera
image_url: https://bs.plantnet.org/image/o/a0c4045351cd7e31ed6c4a338c4fd7f0be5c1a2f


Selected query: rose
basis: genus
common_name: Common-briar
scientific_name: Rosa canina
image_url: https://bs.plantnet.org/image/o/658bff43fbf3b22296b0206094ab2d01852c41e5


Selected query: sunflower
basis: genus
common_name: Sunflower
scientific_name: Helianthus annuus
image_url: https://bs.plantnet.org/image/o/e92bdbc5adc4e91cc0c5aa8e9f102833a28bc6e3


Selected query: strawberry
basis: genus
common_name: Alpine strawberry
scientific_name: Fragaria vesca
image_url: https://bs.plantnet.org/image/o/4f45fd2d82661996f5d5a5613b39bdd1287a56bc


...

Selected query: oak
basis: genus
common_name: Evergreen oak
scientific_name: Quercus rotundifolia
image_url: https://d2seqvvyy3b8p2.cloudfront.net/40ab8e7cdddbe3e78a581b84efa4e893.jpg


Selected query: banana
basis: genus
common_name: Indian-banana
scientific_name: Asimina triloba
image_url: https://bs.plantnet.org/image/o/469e77f18cdeed9d1821c2d88a5a2eab8aeef8cd


Selected query: monstera
basis: genus
common_name: Tarovine
scientific_name: Monstera adansonii
image_url: https://bs.plantnet.org/image/o/38d4346034e89f4e5917357f2bc62cdcd150a3af


Selected query: blueberry
basis: genus
common_name: European blueberry
scientific_name: Vaccinium myrtillus
image_url: https://bs.plantnet.org/image/o/76e8f509a667a1110611d76c7b84d5bbc03ccb1f


PASS: similarity result image card displayable when available


In [45]:
# Cell: Test 5 — bundle image_only contract

def test_bundle_image_only_contract():
    """
    This tests the full PlantDex related bundle.
    If image_only=True, every returned result in every group should have image_url.
    Empty groups are allowed.
    """
    bundle = get_similar_plants_bundle(
        client,
        "tomato",
        max_results_per_group=5,
        image_only=True,
    )

    assert isinstance(bundle, dict)
    assert "groups" in bundle
    assert isinstance(bundle["groups"], dict)

    for group_name, group in bundle["groups"].items():
        assert_similarity_response_shape(group)

        for plant in group["results"]:
            assert plant.get("image_url"), (
                f"{group_name} returned imageless plant despite image_only=True: {plant}"
            )
            assert image_url_is_valid(plant["image_url"]), (
                f"{group_name} returned invalid image URL: {plant['image_url']}"
            )

        print(f"{group_name}: {group['count']} image-bearing results")


run_test("bundle image_only contract", test_bundle_image_only_contract)

same_genus: 5 image-bearing results
same_family_excluding_genus: 5 image-bearing results
same_distribution_excluding_taxonomy: 5 image-bearing results
same_edible_part: 0 image-bearing results
same_growth_habit: 5 image-bearing results
same_growth_form: 0 image-bearing results
same_fruit_color: 0 image-bearing results
PASS: bundle image_only contract


In [46]:
# Cell: Test 6 — render bundle image cards where available

def test_render_bundle_image_cards():
    """
    Visual/manual test:
    render every image-bearing card in a full bundle.
    Empty groups or groups with zero images are fine.
    """
    bundle = get_similar_plants_bundle(
        client,
        "tomato",
        max_results_per_group=3,
        image_only=True,
    )

    rendered_count = 0

    for group_name, group in bundle["groups"].items():
        display(HTML(f"<h3>{html.escape(group_name)}</h3>"))

        if not group["results"]:
            display(HTML("<p>No image results for this group.</p>"))
            continue

        for plant in group["results"]:
            display_plant_image_card(plant, width=240)
            rendered_count += 1

    print("Rendered image cards:", rendered_count)

    # Do not hard-fail if rendered_count == 0, because Trefle image coverage can be sparse.
    # The visual output is the important check here.


run_test("render bundle image cards", test_render_bundle_image_cards)

Rendered image cards: 12
PASS: render bundle image cards


In [47]:
# Cell: Test 7 — missing-image fallback card

def test_missing_image_placeholder_card():
    """
    This tests frontend fallback behavior:
    if a plant lacks image_url, the UI should still render a clean placeholder card.
    """
    imageless_plant = {
        "id": 999999,
        "slug": "example-imageless-plant",
        "common_name": "Example Imageless Plant",
        "scientific_name": "Planta examplea",
        "image_url": None,
        "family": "Exampleaceae",
        "genus": "Example",
    }

    assert get_best_image_url(imageless_plant) is None
    assert has_image(imageless_plant) is False

    display_plant_image_card(imageless_plant)

    print("Rendered placeholder card for imageless plant.")


run_test("missing image placeholder card", test_missing_image_placeholder_card)

Rendered placeholder card for imageless plant.
PASS: missing image placeholder card


In [48]:
# Cell: Test 8 — frontend card contract for image-bearing cards

def test_frontend_image_card_contract():
    """
    This tests the minimum fields the frontend needs to render a clickable plant card.
    """
    payload = client.get("/plants/search", {"q": "tomato"})
    results = payload.get("data") or []

    image_cards = [
        normalize_plant_card(plant)
        for plant in results
        if plant.get("image_url")
    ]

    assert image_cards, "Expected at least one tomato search result with image_url"

    card = image_cards[0]

    assert card.get("id") or card.get("slug"), "Card needs id or slug for routing"
    assert card.get("scientific_name"), "Card needs scientific_name"
    assert card.get("image_url"), "Card needs image_url"
    assert image_url_is_valid(card["image_url"])

    frontend_card = {
        "id": card.get("id"),
        "slug": card.get("slug"),
        "common_name": card.get("common_name"),
        "scientific_name": card.get("scientific_name"),
        "image_url": card.get("image_url"),
    }

    print("Frontend-ready image card:")
    print(frontend_card)

    display_plant_image_card(card)


run_test("frontend image card contract", test_frontend_image_card_contract)

Frontend-ready image card:
{'id': 269338, 'slug': 'solanum-lycopersicum', 'common_name': 'Tomato', 'scientific_name': 'Solanum lycopersicum', 'image_url': 'https://bs.plantnet.org/image/o/400851a79391dbe6f667c66e4bf70299e9921853'}


PASS: frontend image card contract


In [49]:
# Cell: Optional visual gallery from search results

def display_search_image_gallery(query="tomato", max_cards=8):
    """
    Manual visual helper, not really a test.
    Useful for seeing what the frontend search results could look like.
    """
    payload = client.get("/plants/search", {"q": query})
    results = payload.get("data") or []

    cards = [
        normalize_plant_card(plant)
        for plant in results
        if plant.get("image_url")
    ][:max_cards]

    display(HTML(f"<h2>Image search gallery: {html.escape(query)}</h2>"))

    if not cards:
        print("No image cards found.")
        return

    for card in cards:
        display_plant_image_card(card, width=240)


display_search_image_gallery("tomato", max_cards=5)